# Regula Falsi (False Position) Method
> **Numerical Methods for Engineering** | Module 01 - Root Finding | `04_Regula_Falsi.ipynb`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bpatinoa/numerical-methods-for-engineering/blob/main/01-Root-Finding/04_Regula_Falsi.ipynb)

---

## Learning Objectives

After completing this notebook you will be able to:
- Derive the Regula Falsi update as a **bracketed secant line interpolation**.
- Explain why the method **always maintains the root bracket** (unlike the Secant method).
- Identify the **pathological slow-convergence** case and understand the Illinois fix.
- Apply Regula Falsi to problems in general mathematics, chemistry, and telecommunications.
- Compare Bisection and Regula Falsi on both normal and pathological examples.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize':(11,5),'font.size':12,
    'axes.grid':True,'grid.alpha':0.35,'lines.linewidth':2})
print('Libraries loaded OK')

---
## 1. Theoretical Background

### 1.1 Motivation

The Bisection method always halves the interval, regardless of function shape. If $f$ is steep
near $a$ and flat near $b$, the midpoint $c=(a+b)/2$ is a poor estimate of the root.

**Idea:** Replace the midpoint with the $x$-intercept of the **secant line** through the bracket
endpoints — a point that accounts for the function's values, not just the interval geometry.

---

### 1.2 Algorithm

Given a bracket $[a, b]$ with $f(a)\cdot f(b) < 0$, compute:

$$\boxed{c = b - f(b)\,\frac{b - a}{f(b) - f(a)} = \frac{a\,f(b) - b\,f(a)}{f(b) - f(a)}}$$

Then:
- If $f(a)\cdot f(c) < 0$: set $b \leftarrow c$ (root in $[a,c]$)
- Else: set $a \leftarrow c$ (root in $[c,b]$)

The bracket is **always maintained** — this is the key difference from the Secant method.

---

### 1.3 Geometric Interpretation

The update $c$ is the $x$-intercept of the straight line connecting $(a, f(a))$ and $(b, f(b))$.
If the function is approximately linear, this gives a much better estimate than the midpoint.

For a **convex** function (e.g. $f(x) = e^x - C$), the secant line lies *above* $f$, so
$c$ always falls on one side of the root, causing **one endpoint to be frozen** — this leads
to only linear convergence, often slower than Bisection.

---

### 1.4 Convergence

Regula Falsi converges **linearly** ($p = 1$), but unlike Bisection the convergence constant
depends on the function's curvature near the root. In practice:

- For **nearly linear $f$**: Regula Falsi converges faster than Bisection.
- For **highly curved $f$** (e.g. $f(x) = x^{10}-1$): one endpoint gets frozen; convergence
  can be *slower* than Bisection. This is the main pathological case.

**The Illinois Algorithm** fixes this by halving $f(a)$ or $f(b)$ when the same endpoint
is retained twice in a row, forcing the frozen endpoint to move.

---

### 1.5 Pseudocode

```
INPUT:  f, a, b, tol, max_iter
ENSURE: f(a)*f(b) < 0

FOR i = 1 TO max_iter:
    c  <-  (a*f(b) - b*f(a)) / (f(b) - f(a))
    IF |f(c)| < tol  OR  |b - a| < tol:
        RETURN c
    IF f(a)*f(c) < 0:
        b <- c
    ELSE:
        a <- c
END FOR
RETURN c
```

---

### 1.6 Regula Falsi vs. Bisection vs. Secant

| Feature | Bisection | Regula Falsi | Secant |
|---------|-----------|-------------|--------|
| Maintains bracket | Yes | **Yes** | No |
| Update point | Midpoint | Secant intercept | Secant intercept |
| Convergence order | 1 (fixed) | 1 (variable) | 1.618 |
| Derivative needed | No | No | No |
| Risk of divergence | No | No | Yes |
| Pathological case | None | Frozen endpoint | Bad starting points |


In [ ]:
def regula_falsi(f, a, b, tol=1e-8, max_iter=200, verbose=False):
    # Regula Falsi (False Position): always maintains bracket.
    if f(a)*f(b) >= 0:
        raise ValueError('IVT not satisfied: f(a)*f(b) >= 0')
    hist = {'iterates':[], 'errors':[], 'f_values':[], 'a_vals':[], 'b_vals':[]}
    if verbose:
        print(f"{'n':>4}  {'a':>14}  {'b':>14}  {'c':>14}  {'f(c)':>12}  {'|f(c)|':>12}")
        print('-'*76)
    c = a
    for i in range(1, max_iter+1):
        fa, fb = f(a), f(b)
        c  = (a*fb - b*fa) / (fb - fa)
        fc = f(c)
        hist['iterates'].append(c)
        hist['errors'].append(abs(fc))
        hist['f_values'].append(fc)
        hist['a_vals'].append(a)
        hist['b_vals'].append(b)
        if verbose:
            print(f'{i:>4}  {a:>14.8f}  {b:>14.8f}  {c:>14.8f}  {fc:>12.4e}  {abs(fc):>12.4e}')
        if abs(fc) < tol or abs(b-a) < tol:
            break
        if fa*fc < 0:
            b = c
        else:
            a = c
    return c, hist

---
## 2. Examples

### 2.1 General Mathematical Example

Find all real roots of $f(x) = e^x - 3x$.

**Analysis:** $f'(x) = e^x - 3 = 0 \implies x^* = \ln 3 \approx 1.099$ (minimum).
$f(\ln 3) = 3 - 3\ln 3 \approx -0.296 < 0$, so there are **two real roots**.

Plot to identify brackets:
- Root 1: $[0,\, 1]$ since $f(0)=1>0$ and $f(1)=e-3\approx-0.28<0$.
- Root 2: $[1,\, 4]$ since $f(1)<0$ and $f(4)=e^4-12\approx42.6>0$.


In [ ]:
f1 = lambda x: np.exp(x) - 3*x

# Verify brackets
print(f'f(0)={f1(0):.4f}, f(1)={f1(1):.4f}  --> bracket [0,1] for root 1')
print(f'f(1)={f1(1):.4f}, f(4)={f1(4):.4f}  --> bracket [1,4] for root 2')

root1a, h1a = regula_falsi(f1, 0.0, 1.0, tol=1e-10, verbose=True)
print(f'\nRoot 1 = {root1a:.12f}  (f={f1(root1a):.2e})')

root1b, h1b = regula_falsi(f1, 1.0, 4.0, tol=1e-10, verbose=True)
print(f'\nRoot 2 = {root1b:.12f}  (f={f1(root1b):.2e})')

x_range = np.linspace(-0.5, 4.5, 500)
fig, axes = plt.subplots(1, 2, figsize=(13,5))

ax = axes[0]
ax.plot(x_range, f1(x_range), color='steelblue', label=r'$f(x)=e^x-3x$')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.axvline(root1a, color='crimson',    ls=':', lw=2, label=f'Root 1 = {root1a:.5f}')
ax.axvline(root1b, color='darkorange', ls=':', lw=2, label=f'Root 2 = {root1b:.5f}')
ax.scatter([root1a, root1b], [0,0], color=['crimson','darkorange'], s=80, zorder=5)
ax.set(xlabel='x', ylabel='f(x)', ylim=(-1,5), title='Two roots of $e^x-3x$')
ax.legend()

ax = axes[1]
ax.semilogy(range(1,len(h1a['errors'])+1), h1a['errors'], 'o-', color='crimson',    ms=4, label='Root 1')
ax.semilogy(range(1,len(h1b['errors'])+1), h1b['errors'], 's-', color='darkorange', ms=4, label='Root 2')
ax.set(xlabel='Iteration', ylabel='|f(c)| (log)', title='Regula Falsi Convergence')
ax.legend()

plt.tight_layout()
plt.savefig('/sessions/epic-eloquent-mccarthy/mnt/outputs/rf_math.png', dpi=120, bbox_inches='tight')
plt.show()

### 2.2 Chemistry Application - Buffer Solution with Common Ion Effect

**Background:** For a solution containing $C = 0.10$ mol/L acetic acid (CH3COOH) and
$C_a = 0.050$ mol/L sodium acetate (NaOAc), the full charge balance gives:

$$K_a = \frac{x\,(C_a + x - K_w/x)}{C - x + K_w/x}$$

where $x = [\text{H}^+]$, $K_a = 1.8\times10^{-5}$, $K_w = 1\times10^{-14}$.

Rearranging: $\boxed{f(x) = K_a\,(C - x + K_w/x) - x\,(C_a + x - K_w/x) = 0}$

This is more complex than the simple quadratic from Module 01 because the $K_w/x$ terms
couple the acid and water equilibria — it cannot be reduced to a simple quadratic.

**Analytical estimate (Henderson-Hasselbalch):** $\text{pH} \approx \text{p}K_a + \log(C_a/C) = 4.745 - 0.301 = 4.444$
so $x \approx 3.59\times10^{-5}$ mol/L. We bracket: $[10^{-7},\,10^{-3}]$.


In [ ]:
Ka = 1.8e-5
Kw = 1e-14
C  = 0.10    # total acetic acid (mol/L)
Ca = 0.050   # acetate from NaOAc (mol/L)

def f_buffer(x):
    # Full charge-balance equation for acetic acid / sodium acetate buffer
    OH = Kw / x
    return Ka*(C - x + OH) - x*(Ca + x - OH)

a2, b2 = 1e-7, 1e-3
print(f'f({a2:.0e}) = {f_buffer(a2):.4e}  (should be > 0)')
print(f'f({b2:.0e}) = {f_buffer(b2):.4e}  (should be < 0)')

root2, h2 = regula_falsi(f_buffer, a2, b2, tol=1e-14, verbose=False)

pH_num = -np.log10(root2)
pH_HH  = -np.log10(Ka) + np.log10(Ca/C)
print(f'\n[H+] (Regula Falsi)        : {root2:.6e} mol/L')
print(f'pH   (Regula Falsi)        : {pH_num:.5f}')
print(f'pH   (Henderson-Hasselbalch): {pH_HH:.5f}')
print(f'Difference                 : {abs(pH_num-pH_HH):.5f} units')
print(f'Iterations                 : {len(h2["iterates"])}')

x_range = np.logspace(-7, -3, 600)
fig, axes = plt.subplots(1, 2, figsize=(13,5))

ax = axes[0]
ax.semilogx(x_range, [f_buffer(x) for x in x_range], color='teal')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.axvline(root2, color='darkorange', ls=':', lw=2,
           label=f'[H+] = {root2:.3e}  (pH = {pH_num:.3f})')
ax.scatter([root2], [0], color='darkorange', s=80, zorder=5)
ax.set(xlabel='[H+] (mol/L)', ylabel='f([H+])',
       title='Buffer Equilibrium: 0.10 M HAc + 0.05 M NaOAc')
ax.legend(fontsize=9)

ax = axes[1]
ax.semilogy(range(1,len(h2['errors'])+1), h2['errors'], 'o-', color='teal', ms=4)
ax.set(xlabel='Iteration', ylabel='|f(c)| (log)', title='Convergence - Buffer Chemistry')

plt.tight_layout()
plt.savefig('/sessions/epic-eloquent-mccarthy/mnt/outputs/rf_chem.png', dpi=120, bbox_inches='tight')
plt.show()

### 2.3 Telecommunications Application - Maximum Radar Detection Range

**Background:** The radar range equation gives the received power from a target at distance $R$:

$$P_r(R) = \frac{P_t\,G^2\,\lambda^2\,\sigma}{(4\pi)^3\,R^4}\,e^{-2\alpha R}$$

where $\alpha$ [Np/m] accounts for **atmospheric absorption**. Setting $P_r(R^*) = P_\text{min}$:

$$\boxed{f(R) = R^4\,e^{2\alpha R} - \frac{P_t\,G^2\,\lambda^2\,\sigma}{(4\pi)^3\,P_\text{min}} = 0}$$

This is **transcendental** (power law × exponential) — no closed-form solution exists.
Since $f(R)$ is monotonically increasing, a bracket $[R_1, R_2]$ is easy to find. Regula
Falsi is ideal: it keeps the bracket while converging faster than Bisection.

**Parameters (S-band air traffic control radar):**

| Parameter | Value |
|-----------|-------|
| $P_t$ | 1.4 MW |
| $G$ | 33 dBi ($\approx 1995$ linear) |
| $\lambda = c/f$ | 3e8/2.8e9 = 0.1071 m |
| $\sigma$ | 1 m² (small aircraft RCS) |
| $\alpha$ | $3\times10^{-6}$ Np/m (tropospheric absorption) |
| $P_\text{min}$ | $10^{-14}$ W ($-140$ dBm) |


In [ ]:
Pt  = 1.4e6          # W  (transmit power)
G   = 10**(33/10)    # linear gain (33 dBi)
c_light = 3e8
freq_r  = 2.8e9
lam_r   = c_light / freq_r   # 0.1071 m
sigma   = 1.0        # m^2  (target RCS)
alpha_r = 3e-6       # Np/m (atmospheric absorption)
P_min   = 1e-14      # W  (-140 dBm receiver sensitivity)

# Radar constant K = Pt*G^2*lam^2*sigma / ((4pi)^3 * P_min)
K_radar = Pt * G**2 * lam_r**2 * sigma / ((4*np.pi)**3 * P_min)
print(f'Radar constant K = {K_radar:.4e} m^4')

# f(R) = R^4 * exp(2*alpha*R) - K_radar = 0
f_radar = lambda R: R**4 * np.exp(2*alpha_r*R) - K_radar

# Find bracket (in km, convert to m)
R_lo, R_hi = 1e5, 3e5   # 100 km to 300 km
print(f'f({R_lo/1e3:.0f} km) = {f_radar(R_lo):.3e}')
print(f'f({R_hi/1e3:.0f} km) = {f_radar(R_hi):.3e}')

root3, h3 = regula_falsi(f_radar, R_lo, R_hi, tol=1e-3, max_iter=200, verbose=False)
print(f'\nMax radar detection range: {root3/1e3:.2f} km')
print(f'Check: f(R*) = {f_radar(root3):.4e}  (should be ~0)')
print(f'Iterations: {len(h3["iterates"])}')

R_vals = np.linspace(R_lo, R_hi, 800)
Pr_vals = Pt*G**2*lam_r**2*sigma / ((4*np.pi)**3 * R_vals**4) * np.exp(-2*alpha_r*R_vals)

fig, axes = plt.subplots(1, 2, figsize=(13,5))

ax = axes[0]
ax.semilogy(R_vals/1e3, Pr_vals, color='royalblue', label='$P_r(R)$')
ax.axhline(P_min, color='crimson', ls='--', label=f'$P_{{min}}$ = {P_min:.0e} W')
ax.axvline(root3/1e3, color='darkorange', ls=':', lw=2,
           label=f'$R^*$ = {root3/1e3:.1f} km')
ax.scatter([root3/1e3], [P_min], color='darkorange', s=80, zorder=5)
ax.set(xlabel='Range R (km)', ylabel='Received Power (W)',
       title='Radar Detection Range (S-band ATC Radar)')
ax.legend()

ax = axes[1]
ax.semilogy(range(1,len(h3['errors'])+1), h3['errors'], 'o-', color='royalblue', ms=4)
ax.set(xlabel='Iteration', ylabel='|f(c)| (log)', title='Regula Falsi Convergence - Radar')

plt.tight_layout()
plt.savefig('/sessions/epic-eloquent-mccarthy/mnt/outputs/rf_telecom.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. Pathological Case: Frozen Endpoint

For a **convex function** on a bracket where one side is very steep, Regula Falsi freezes
one endpoint — convergence can become slower than Bisection.

**Classic example:** $f(x) = x^{10} - 1$ on $[0,\,1.5]$.

The function is very flat near $x=0$ and very steep near $x=1.5$, so the secant line
always intersects far from the root $r=1$. The left endpoint $a=0$ **never moves**.


In [ ]:
f_path = lambda x: x**10 - 1
r_path = 1.0

_, h_bis_p = (lambda: (__import__('functools').reduce(
    lambda acc,_: [acc[0] if f_path(acc[0])*f_path((acc[0]+acc[1])/2)<0 else (acc[0]+acc[1])/2,
                   acc[1] if f_path(acc[0])*f_path((acc[0]+acc[1])/2)<0 else (acc[0]+acc[1])/2],
    range(40), [0.0, 1.5])
    , None))()  if False else (None, None)

# Simple bisection for comparison
def bisection_simple(f, a, b, tol=1e-10, max_iter=100):
    errs=[]
    for _ in range(max_iter):
        c=(a+b)/2; errs.append(abs(f(c)))
        if abs(f(c))<tol: break
        if f(a)*f(c)<0: b=c
        else: a=c
    return c, errs

root_bis_p, errs_bis = bisection_simple(f_path, 0.0, 1.5, tol=1e-10)
root_rf_p,  h_rf_p   = regula_falsi(f_path, 0.0, 1.5, tol=1e-10, max_iter=500)

print(f'Bisection iterations : {len(errs_bis)}')
print(f'Regula Falsi iters   : {len(h_rf_p["iterates"])}')
print('** Regula Falsi is SLOWER on this convex function! **')

# Frozen endpoint: show that a stays at 0
a_vals = h_rf_p['a_vals']
frozen = sum(1 for av in a_vals if abs(av) < 1e-10)
print(f'Left endpoint a=0 frozen for {frozen}/{len(a_vals)} iterations')

fig, axes = plt.subplots(1, 2, figsize=(13,5))

x_p = np.linspace(0, 1.5, 500)
ax = axes[0]
ax.plot(x_p, f_path(x_p), color='steelblue')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.axvline(1.0, color='crimson', ls=':', lw=2, label='Root r=1')
ax.set(xlabel='x', ylabel='f(x)', title=r'Pathological case: $f(x)=x^{10}-1$')
ax.legend()

ax = axes[1]
ax.semilogy(range(1,len(errs_bis)+1),              errs_bis,
            'o--', color='steelblue', ms=4, label=f'Bisection ({len(errs_bis)} iters)')
ax.semilogy(range(1,len(h_rf_p['errors'])+1), h_rf_p['errors'],
            's-',  color='darkorange', ms=4, label=f'Regula Falsi ({len(h_rf_p["iterates"])} iters)')
ax.set(xlabel='Iteration', ylabel='|f(c)| (log)',
       title='Regula Falsi slower than Bisection (frozen endpoint)')
ax.legend()

plt.tight_layout()
plt.savefig('/sessions/epic-eloquent-mccarthy/mnt/outputs/rf_pathological.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Student Exercises

---

**Exercise 1 - Multiple brackets**
Find all roots of $g(x) = \sin(x) - x/2$ in $[-2\pi, 2\pi]$ using Regula Falsi with $\varepsilon = 10^{-8}$.
*Hint*: Plot first; note the root at $x=0$ has even multiplicity — how does Regula Falsi behave?

---

**Exercise 2 - Carbonate system pH (Chemistry)**
For a 0.01 M $\text{CO}_2$ solution with $K_{a1} = 4.3\times10^{-7}$, $K_{a2} = 4.8\times10^{-11}$:
$$f([H^+]) = [H^+]^3 + K_{a1}[H^+]^2 - (K_{a1}K_w + K_{a1}C)[H^+] - 2K_{a1}K_{a2}C = 0$$
Find the pH using Regula Falsi. Compare with the approximation $\text{pH}\approx\frac{1}{2}(\text{p}K_{a1} - \log C)$.

---

**Exercise 3 - Radar cross-section estimation (Telecom)**
Given a radar system identical to Example 2.3, but now $R^*=150$ km is observed.
Back-solve to find the **target RCS $\sigma$** (in m²) that would produce this range.
Set up $f(\sigma)=0$ and apply Regula Falsi with bracket $[0.01, 100]$ m².

---

**Exercise 4 - Illinois Algorithm**
Implement the **Illinois modification** of Regula Falsi, which halves the function value
at the retained endpoint when it does not change for two consecutive iterations:

```
IF same endpoint retained twice:
    f(a) <- f(a) / 2   [or f(b) <- f(b)/2]
```

Compare the Illinois method vs. standard Regula Falsi on the pathological case $f(x)=x^{10}-1$.

---

**Exercise 5 - Method selection**
Given the four methods studied (Bisection, Newton-Raphson, Secant, Regula Falsi), recommend
the **best method** for each scenario and justify:
a) $f(x) = \sqrt[3]{x}$ near $x=0$ (root at $x=0$, $f'(0)$ undefined)
b) $f(x) = e^{-x^2} - 0.5$ (smooth, derivative easy, good initial guess available)
c) A black-box numerical simulation where only $f(x)$ values are available
d) A function with two roots close together in a known interval


---
## 5. References

1. **Chapra, S. C., & Canale, R. P.** (2015). *Numerical Methods for Engineers* (7th ed., pp. 112-120).
   McGraw-Hill Education.
   *Section 5.2 introduces Regula Falsi, compares it to Bisection, and includes the frozen-endpoint
   discussion that motivates the Illinois fix in Exercise 4.*

2. **Burden, R. L., Faires, J. D., & Burden, A. M.** (2016). *Numerical Analysis* (10th ed., pp. 51-55).
   Cengage Learning.
   *Section 2.1 provides a theoretical bound on the convergence rate and proves that Regula Falsi
   is linearly convergent, with rate depending on the curvature of $f$ near the root.*

3. **Hamming, R. W.** (1987). *Numerical Methods for Scientists and Engineers* (2nd ed., pp. 62-70).
   Dover Publications.
   *Chapter 4 analyses bracketing methods from a practitioner's viewpoint, including the
   pathological convergence case and the Illinois algorithm fix described in Exercise 4.*

4. **Atkins, P., & de Paula, J.** (2014). *Physical Chemistry* (10th ed., pp. 163-175).
   Oxford University Press.
   *Source for the buffer equilibrium chemistry in Example 2.2. Section 6C covers weak acid
   equilibria and the Henderson-Hasselbalch approximation used as a reference solution.*

5. **Skolnik, M. I.** (2001). *Introduction to Radar Systems* (3rd ed., pp. 1-25).
   McGraw-Hill.
   *The radar range equation used in Example 2.3 is derived in Chapter 1. Table 1-1 provides
   typical ATC radar parameters consistent with the values used in this notebook.*
